# Phase 1 — Training & Preliminary Validation

Trains the meta-encoder and evaluates the two foundational success criteria:

| Criterion | Metric | Target |
|-----------|--------|--------|
| C1 | Profile Reconstruction R² | ≥ 0.70 |
| C2 | Geometric Consistency (Spearman ρ) | per-layer > 0.50, mean > 0.65 |

**Workflow:** run cells top-to-bottom.  
To skip training (checkpoint already exists), run sections 0–1 then jump to section 3.

## 0. Colab Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/jacobposchl/trainable-circuits'
REPO_DIR = '/content/model_interpretability'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print(f'Repo:     {REPO_DIR}')
print(f'Data:     {DATA_DIR}')
print(f'Checkpts: {CKPT_DIR}')

## 1. Configuration

Edit **only this cell** to switch between model architectures.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  SELECT MODEL  —  uncomment one line
# ─────────────────────────────────────────────────────────────────────────────
MODEL_CONFIG = CONFIG_DIR + '/models/resnet18.yaml'
# MODEL_CONFIG = CONFIG_DIR + '/models/resnet34.yaml'
# MODEL_CONFIG = CONFIG_DIR + '/models/resnet50.yaml'

print(f'Using config: {MODEL_CONFIG}')

## 2. Training

Loads the selected config, trains if no checkpoint exists, then prints a summary.

In [ ]:
import yaml
import torch
from training import Phase1Trainer

with open(MODEL_CONFIG) as f:
    config = yaml.safe_load(f)

# Redirect paths to Colab / Drive locations
config['data']['data_dir']          = DATA_DIR
config['logging']['checkpoint_dir'] = CKPT_DIR + '/' + config['experiment']['name']

checkpoint_path = config['logging']['checkpoint_dir'] + '/best.pt'
device          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Model      : {config['experiment']['name']}")
print(f"Checkpoint : {checkpoint_path}")
print(f"Device     : {device}")

In [ ]:
if os.path.exists(checkpoint_path):
    print(f'Checkpoint found — skipping training.')
    print(f'Delete {checkpoint_path} to retrain from scratch.')
else:
    print('Starting training...')
    trainer = Phase1Trainer(config)
    trainer.train()
    print('Training complete.')

In [ ]:
if os.path.exists(checkpoint_path):
    ckpt    = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    metrics = ckpt.get('val_metrics', {})
    r2      = metrics.get('r2',       float('nan'))
    rho     = metrics.get('mean_rho', float('nan'))
    epoch   = ckpt.get('epoch', -1) + 1
    status  = 'PASS' if r2 >= 0.7 and rho >= 0.65 else 'FAIL'
    print(f"Experiment : {config['experiment']['name']}")
    print(f"Epoch      : {epoch}")
    print(f"R\u00b2         : {r2:.4f}  (target \u2265 0.70)  {'\u2713' if r2 >= 0.7 else '\u2717'}")
    print(f"Mean \u03c1     : {rho:.4f}  (target > 0.65)  {'\u2713' if rho >= 0.65 else '\u2717'}")
    print(f"Status     : {status}")
else:
    print('No checkpoint found. Run the training cell above first.')

## 3. Criterion 1 — Profile Reconstruction R²

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from evaluation.circuit_analysis import CircuitAnalyzer, load_checkpoint
from data.cifar import get_standard_loaders

backbone, meta_encoder, info_loss = load_checkpoint(config, checkpoint_path, device)

_, val_loader = get_standard_loaders(
    data_dir   = config['data']['data_dir'],
    batch_size = config['data'].get('batch_size', 256),
    num_workers= config['data'].get('num_workers', 4),
    augment    = False,
)

In [ ]:
MAX_SAMPLES = 2000

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
print(f'Collecting representations (max {MAX_SAMPLES} samples)...')
data = analyzer.collect_representations(max_samples=MAX_SAMPLES)

flow_targets = data['flow_targets']
z_list       = data['z_list']
labels       = data['labels']
N = labels.shape[0]
L = len(z_list)
print(f'Collected {N} samples across {L} layers.')

In [ ]:
from evaluation.metrics import rich_profile_reconstruction_r2

MAX_PAIRS    = 50_000
idx_a, idx_b = torch.triu_indices(N, N, offset=1)
if idx_a.shape[0] > MAX_PAIRS:
    perm = torch.randperm(idx_a.shape[0])[:MAX_PAIRS]
    idx_a, idx_b = idx_a[perm], idx_b[perm]

rich_preds, rich_trues = [], []
with torch.no_grad():
    for l in range(L):
        target_l = (flow_targets[l][idx_a] * flow_targets[l][idx_b]).numpy()
        z_product = (z_list[l][idx_a] * z_list[l][idx_b]).to(device)
        pred_l = info_loss.regressors[l](z_product).cpu().numpy()
        rich_preds.append(pred_l)
        rich_trues.append(target_l)

rich = rich_profile_reconstruction_r2(rich_preds, rich_trues)
print(f"Overall R^2 : {rich['r2']:.4f}")
print("Target      : >= 0.70")
print(f"Status      : {'PASS' if rich['passes'] else 'FAIL'}")


In [ ]:
per_layer_r2 = rich['per_layer_r2']

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if r >= 0.7 else 'coral' for r in per_layer_r2]
ax.bar(range(L), per_layer_r2, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0.7, color='red', linestyle='--', linewidth=1.2, label='Target (R^2=0.70)')
ax.set_xlabel('Layer')
ax.set_ylabel('R^2')
ax.set_xticks(range(L))
ax.set_xticklabels([f'L{l+1}' for l in range(L)])
ax.set_ylim(0, 1.05)
ax.set_title(f"C1 - Per-Layer Profile Reconstruction R^2 (overall={rich['r2']:.3f})")
ax.legend()
fig.tight_layout()
plt.show()


## 4. Criterion 2 — Geometric Consistency

In [ ]:
from evaluation import geometric_consistency

z_sims = np.zeros((idx_a.shape[0], L))
for l in range(L):
    z_a = z_list[l][idx_a].numpy()
    z_b = z_list[l][idx_b].numpy()
    z_sims[:, l] = (z_a * z_b).sum(axis=1)

gc = geometric_consistency(z_sims, true_sims, L)
print(f"Mean \u03c1 : {gc['mean_rho']:.4f}  (target > 0.65)  {'\u2713' if gc['mean_rho'] > 0.65 else '\u2717'}")
print()
print('Per-layer \u03c1:')
for l, rho in enumerate(gc['per_layer_rho']):
    mark = '\u2713' if rho > 0.5 else '\u2717'
    print(f'  L{l+1}: {rho:.4f}  {mark}')
print(f"\nStatus : {'PASS \u2713' if gc['passes'] else 'FAIL \u2717'}")

In [ ]:
per_layer_rho = gc['per_layer_rho']

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if r > 0.5 else 'coral' for r in per_layer_rho]
ax.bar(range(L), per_layer_rho, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0.50, color='orange',    linestyle='--', linewidth=1.2, label='Per-layer target (\u03c1=0.50)')
ax.axhline(0.65, color='red',       linestyle='--', linewidth=1.2, label='Mean target (\u03c1=0.65)')
ax.axhline(gc['mean_rho'], color='steelblue', linestyle='-', linewidth=1.5,
           label=f"Mean \u03c1={gc['mean_rho']:.3f}")
ax.set_xlabel('Layer')
ax.set_ylabel('Spearman \u03c1')
ax.set_xticks(range(L))
ax.set_xticklabels([f'L{l+1}' for l in range(L)])
ax.set_ylim(0, 1.05)
ax.set_title('C2 \u2014 Per-Layer Geometric Consistency (Spearman \u03c1)')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
from evaluation import plot_per_layer_umap

z_np      = [z.numpy() for z in z_list]
labels_np = labels.numpy()

fig = plot_per_layer_umap(z_np, labels_np, max_samples=1500)
fig.suptitle(f"Per-Layer Circuit Space \u2014 {config['experiment']['name']}", fontsize=13)
plt.show()